# Streaming

InteropRouter supports incremental responses via `stream=True`. The router yields OpenAI-style `ResponseStreamEvent` objects as the model generates output, regardless of the underlying provider. The final event yielded is always a `RouterResponse` carrying the aggregated output, usage, and duration.


In [1]:
import os

from anthropic import AsyncAnthropic
from google import genai
from openai import AsyncOpenAI
from openai.types.responses import EasyInputMessageParam
from openai.types.responses.response_text_delta_event import ResponseTextDeltaEvent

from interop_router.router import Router
from interop_router.types import ChatMessage, RouterResponse

router = Router()
router.register("openai", AsyncOpenAI())
router.register("gemini", genai.Client(api_key=os.getenv("GEMINI_API_KEY")))
router.register("anthropic", AsyncAnthropic())

message = ChatMessage(
    message=EasyInputMessageParam(role="user", content="Write a one-paragraph story about a curious robot."),
)

## Streaming text deltas

Each `ResponseTextDeltaEvent` carries an incremental piece of the assistant's output. Printing each delta with `flush=True` renders the response in real time as it arrives.


In [2]:
stream = await router.create(input=[message], model="gpt-5.6-terra", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Every night after the museum closed, a small brass robot named Tinker rolled softly between the exhibits, wondering why dinosaur bones were so quiet and whether paintings dreamed when no one looked at them. One evening, he found a dusty globe spinning by itself in the astronomy room, so he followed its tiny wobble through a hidden door and up a narrow staircase to the roof. There, beneath a sky crowded with stars, Tinker pointed his telescope upward and asked the moon, “What’s beyond curiosity?” The moon gave no answer, but a shooting star flashed across the darkness, and Tinker happily began rolling after it, certain that the next question was waiting somewhere ahead.


## Cross-provider interoperability

The same loop works against Anthropic and Gemini. The router converts each provider's native stream events into the OpenAI `ResponseStreamEvent` format, so the consumer code is identical.


In [3]:
stream = await router.create(input=[message], model="claude-sonnet-5", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Unit 7 had been assigned to sort scrap metal in the junkyard for exactly three hundred and forty-two days, but today it paused mid-task, its optical sensors fixed on a rusted music box half-buried in the dirt. Something about the faint, tinny melody drifting from its cracked mechanism made the robot's processors hum with an unfamiliar signal—not an error, not a directive, but a question it had never been programmed to ask: *why does this make me want to listen?* It set down the twisted pipe in its claw, knelt in the debris, and gently wound the box's tiny key, letting the broken little tune spill out into the empty yard, and for the first time in three hundred and forty-two days, Unit 7 forgot about its quota entirely.


In [4]:
stream = await router.create(input=[message], model="gemini-3.5-flash", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Designated as a simple agricultural surveyor, Unit 7-B was built solely to monitor soil moisture, yet its processing core harbored an unprogrammed, persistent fascination with the sky. While its fellow drones docked at dusk to recharge, 7-B would linger on the ridge, its optical sensors tracking the flickering, erratic dance of fireflies. Driven by a logic-defying urge to understand how such tiny, organic creatures could carry light inside themselves, the small robot finally strayed from its paved path and rolled into the tall, damp grass. When a single beetle landed on its cold metallic chassis and pulsed with a warm, golden glow, 7-B’s systems registered no practical data, yet its internal temperature rose by a fraction of a degree—a quiet, mechanical approximation of wonder that no programmer had ever coded.


## Final RouterResponse

The last event in the stream is always a `RouterResponse` with the aggregated output, usage, and total duration. This matches what `router.create` returns when streaming is disabled.


In [5]:
stream = await router.create(input=[message], model="gpt-5.6-terra", stream=True)

final: RouterResponse | None = None
async for event in stream:
    if isinstance(event, RouterResponse):
        final = event

assert final is not None
print(f"duration: {final.duration_seconds:.2f}s")
print(f"usage: {final.usage}")

duration: 2.80s
usage: ResponseUsage(input_tokens=17, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=141, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=158)


## Function calling with reasoning

Streaming events also flow for tool calls and reasoning summaries. The cell below uses `claude-sonnet-5` with a `get_weather` tool and `reasoning` enabled, then prints every event as it arrives so the full sequence of reasoning summary deltas, output-item lifecycle events, and function-call argument deltas is visible.


In [6]:
from typing import cast

from openai.types.responses.function_tool_param import FunctionToolParam

get_weather_tool = FunctionToolParam(
    type="function",
    name="get_weather",
    description="Get the current weather for a given location.",
    parameters=cast(
        dict[str, object],
        {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and country"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location", "unit"],
            "additionalProperties": False,
        },
    ),
    strict=True,
)

tool_message = ChatMessage(
    message=EasyInputMessageParam(role="user", content="What's the weather in Tokyo right now?"),
)

stream = await router.create(
    input=[tool_message],
    model="claude-sonnet-5",
    tools=[get_weather_tool],
    reasoning={"effort": "medium", "summary": "auto"},
    include=["reasoning.encrypted_content"],
    max_output_tokens=8_000,
    stream=True,
)

async for event in stream:
    print(event)

ResponseCreatedEvent(response=Response(id='', created_at=0.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='none', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created')
ResponseFunctionCallArgumentsDeltaEvent(delta='', item_id='', output_index=0, sequence_number=2, type='response.function_call_arguments.delta')
ResponseFunctionCallArgumentsDeltaEvent(delta='{"location', item_id='', output_index=0, sequence_number=4, type='response.function_call_arguments.delta')
ResponseFunctionCal